<a href="https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signals

- `age_days`: number of days from the last update to the decision date.
- `decline_ratio`: `(H1 − H2) / H1`, the ratio of the impressions drop between the two halves of month 3.


- `age_norm` = `min(age_days / 365, 1)`
- `decline_norm` = `min(max(decline_ratio, 0), 1)`

### Score

`score = w1 × age_norm + w2 × decline_norm`

In [1]:
import duckdb, pandas as pd
from getpass import getpass
from huggingface_hub import login

TOKEN = getpass("HF token: ")
login(token=TOKEN)

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
SRC_DAILY   = f"{BASE}/fact_content_daily_performance/**/*.parquet"
SRC_CLIENTS = f"{BASE}/dim_clients.parquet"
SRC_CONTENT = f"{BASE}/dim_content.parquet"

START, END  = "2026-03-01", "2026-03-31"
MIN_IMPR_M3 = 100   # الحد الأدنى للـ impressions في شهر 3

con.execute(f"""
CREATE OR REPLACE TEMP TABLE content_slice AS
SELECT d.client_hash_id, d.content_hash_id
FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
JOIN read_parquet('{SRC_CLIENTS}') c USING (client_hash_id)
WHERE d.month = '2026-03'
  AND d.report_date BETWEEN DATE '{START}' AND DATE '{END}'
  AND CAST(c.gsc_data_start AS DATE) < DATE '{START}'
GROUP BY d.client_hash_id, d.content_hash_id
HAVING SUM(d.gsc_impressions) >= {MIN_IMPR_M3}
""")

slice_df = con.execute("SELECT * FROM content_slice").df()
print(len(slice_df))


HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101168


In [2]:
slice_df

,client_hash_id,content_hash_id
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1,client_73cda7b4e4f265ea,content_905aa32a0230694e
2,client_73cda7b4e4f265ea,content_05434271b257bb68
3,client_73cda7b4e4f265ea,content_d056587ff7faca0c
4,client_73cda7b4e4f265ea,content_2662845f598544ef
...,...,...
101163,client_3ffa76342f366962,content_48c9b57be92bb11b
101164,client_3ffa76342f366962,content_52c98b3f26951f3b
101165,client_1a730cb2640a1abf,content_051c8125a3d20e83
101166,client_1a730cb2640a1abf,content_45155b648fff917d


In [4]:
stale_df = con.execute(f"""
SELECT
    d.content_hash_id AS content,
    s.client_hash_id AS client,
    DATE_DIFF(
        'day',
        CAST(
            CASE
                WHEN d.content_updated_date IS NOT NULL
                     AND d.content_updated_date <= DATE '2026-03-31'
                THEN d.content_updated_date
                ELSE d.content_created_date
            END AS DATE
        ),
        DATE '2026-03-31'
    ) AS stale_days
FROM read_parquet('{SRC_CONTENT}') d
JOIN slice_df s USING (content_hash_id)
WHERE CASE
    WHEN d.content_updated_date IS NOT NULL
         AND d.content_updated_date <= DATE '2026-03-31'
    THEN d.content_updated_date
    ELSE d.content_created_date
END IS NOT NULL
""").df()

In [5]:
stale_df

,content,client,stale_days
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,396
1,content_905aa32a0230694e,client_73cda7b4e4f265ea,396
2,content_05434271b257bb68,client_73cda7b4e4f265ea,396
3,content_d056587ff7faca0c,client_73cda7b4e4f265ea,396
4,content_2662845f598544ef,client_73cda7b4e4f265ea,396
...,...,...,...
101163,content_48c9b57be92bb11b,client_3ffa76342f366962,425
101164,content_52c98b3f26951f3b,client_3ffa76342f366962,245
101165,content_051c8125a3d20e83,client_1a730cb2640a1abf,3
101166,content_45155b648fff917d,client_1a730cb2640a1abf,1


In [18]:
content_id = "content_8d5d1221c84f8050"
stale_df[stale_df["content"] == content_id]

,content,client,stale_days
79596,content_8d5d1221c84f8050,client_fef1a8f436438636,193


In [6]:
position_df = con.execute(f"""
SELECT
    d.content_hash_id AS content,
    d.client_hash_id AS client,
    AVG(d.gsc_avg_position) AS avg_position
FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
JOIN slice_df s
    USING (content_hash_id, client_hash_id)
WHERE d.month = '2026-03'
  AND d.report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
GROUP BY d.content_hash_id, d.client_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
label_df = con.execute(f"""
WITH monthly AS (
    SELECT
        d.content_hash_id AS content,
        d.client_hash_id AS client,
        AVG(CASE WHEN d.month = '2026-03'
                 THEN d.gsc_impressions END) AS avg_march,
        AVG(CASE WHEN d.month = '2026-04'
                 THEN d.gsc_impressions END) AS avg_april
    FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
    JOIN slice_df s USING (content_hash_id, client_hash_id)
    WHERE d.month IN ('2026-03', '2026-04')
    GROUP BY d.content_hash_id, d.client_hash_id
)
SELECT
    content,
    client,
    CASE
        WHEN avg_march IS NULL OR avg_march = 0 OR avg_april IS NULL
            THEN NULL
        WHEN (avg_april - avg_march) / avg_march > 0.20
            THEN 'Up'
        WHEN (avg_april - avg_march) / avg_march < -0.20
            THEN 'Down'
        ELSE 'Stable'
    END AS label
FROM monthly
WHERE avg_march IS NOT NULL
  AND avg_march > 0
  AND avg_april IS NOT NULL
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
label_df

,content,client,label
0,content_905aa32a0230694e,client_73cda7b4e4f265ea,Down
1,content_05434271b257bb68,client_73cda7b4e4f265ea,Up
2,content_712c365258cee05c,client_73cda7b4e4f265ea,Up
3,content_476c37c366920c1b,client_73cda7b4e4f265ea,Down
4,content_3dba50ae010f3f30,client_73cda7b4e4f265ea,Down
...,...,...,...
101163,content_599d73b602d07483,client_3ffa76342f366962,Up
101164,content_908a53e9eec3b6db,client_3ffa76342f366962,Down
101165,content_8d4c42e5457c9e2e,client_3ffa76342f366962,Stable
101166,content_708b842cd69e0b38,client_3ffa76342f366962,Down


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.